<!-- NB_DOC_INTRO_V1 -->
# MLOps — Monitoring et detection de drift

> 📚 **Doc thematique** : [docs/08_MLOPS.md](docs/08_MLOPS.md) (MLOps)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Un modele ML deploye **se degrade avec le temps**, parfois silencieusement. Trois causes principales :

1. **Data drift** (`P(X)` change) — la distribution des features evolue (saisonnalite, nouveaux clients, capteur degrade)
2. **Concept drift** (`P(Y|X)` change) — la relation features → cible evolue (changement de comportement utilisateur, contexte covid, etc.)
3. **Schema drift** — colonnes ajoutees/supprimees/typees autrement par un upstream (souvent silencieux et fatal)

Sans monitoring, on s'en rend compte **trop tard** (chute des KPIs business, alerte d'un client, audit reglementaire). **Le monitoring detecte ca avant le DSI.**

Ce notebook couvre :
- Implementations **from-scratch** des tests stat principaux (KS, Chi-square, PSI, JS divergence)
- **Evidently AI** pour reports HTML automatises
- **River** pour online learning + drift natif (streaming)
- **Patterns** d'alerting et de retraining
- **Anti-pattern** classique : alerter sur chaque p-value < 0.05 → spam

Versions visees : `evidently >= 0.4`, `river >= 0.21`, `scipy >= 1.11`, `scikit-learn >= 1.4`.

## Plan

1. Installation + simulation d'un dataset avec drift
2. **Tests univaries** : KS (quanti), Chi-square (quali), PSI (Population Stability Index)
3. **Tests multivaries** : MMD (Maximum Mean Discrepancy), Domain classifier
4. **Drift sur la cible** (label drift)
5. **Concept drift** (degradation de la perf)
6. **Evidently AI** : reports auto + tests suite
7. **River** : online learning + ADWIN drift detector
8. **Patterns d'architecture monitoring**
9. Pieges et anti-patterns
10. References


## 1. Installation et simulation d'un dataset avec drift

On simule un cas reel : pendant le temps `t=0`, distribution stable. A `t=T_drift`, certaines features driftent (loi gaussienne decalee, categorielles avec frequences modifiees).


In [ ]:
# pip install -q scipy scikit-learn numpy pandas matplotlib seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# === Generation reference + production (drift) ===
def generate(n, mean_x1=0, prob_cat_A=0.5, seed=None):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "x1":      rng.normal(loc=mean_x1, scale=1.0, size=n),       # numerique
        "x2":      rng.exponential(scale=2.0, size=n),                # numerique skew
        "cat":     rng.choice(["A", "B", "C"], size=n, p=[prob_cat_A, (1-prob_cat_A)/2, (1-prob_cat_A)/2]),
        "constant": rng.normal(0, 0.1, size=n),                       # feature stable
    })

# Reference : entrainee sur cette distribution
ref = generate(2000, mean_x1=0, prob_cat_A=0.5, seed=SEED)

# Production t=0 : memes parametres -> pas de drift (controle)
prod_no_drift = generate(2000, mean_x1=0, prob_cat_A=0.5, seed=SEED + 1)

# Production t=T : DRIFT SUR x1 et cat
prod_drifted = generate(2000, mean_x1=0.7, prob_cat_A=0.3, seed=SEED + 2)

print(f"Reference : {ref.shape}, mean(x1)={ref['x1'].mean():.3f}, P(A)={ref['cat'].value_counts(normalize=True)['A']:.3f}")
print(f"Prod NO drift : {prod_no_drift.shape}, mean(x1)={prod_no_drift['x1'].mean():.3f}, P(A)={prod_no_drift['cat'].value_counts(normalize=True)['A']:.3f}")
print(f"Prod drifted  : {prod_drifted.shape}, mean(x1)={prod_drifted['x1'].mean():.3f}, P(A)={prod_drifted['cat'].value_counts(normalize=True)['A']:.3f}")

### Visualisation du drift


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# x1 : gaussienne decalee
for df, label, color in [(ref, "ref", "C0"), (prod_no_drift, "prod (no drift)", "C2"),
                          (prod_drifted, "prod (drifted)", "C3")]:
    axes[0].hist(df["x1"], bins=40, alpha=0.4, label=label, color=color)
axes[0].set(title="x1 (drift moyenne)", xlabel="x1")
axes[0].legend()

# x2 : pas de drift
for df, label, color in [(ref, "ref", "C0"), (prod_no_drift, "prod (no drift)", "C2"),
                          (prod_drifted, "prod (drifted)", "C3")]:
    axes[1].hist(df["x2"], bins=40, alpha=0.4, label=label, color=color)
axes[1].set(title="x2 (pas de drift)", xlabel="x2")
axes[1].legend()

# cat : categorielle avec frequences modifiees
pd.DataFrame({
    "ref":          ref["cat"].value_counts(normalize=True),
    "prod_no_drift": prod_no_drift["cat"].value_counts(normalize=True),
    "prod_drifted":  prod_drifted["cat"].value_counts(normalize=True),
}).plot(kind="bar", ax=axes[2])
axes[2].set(title="cat (drift frequences)", ylabel="probability")
plt.tight_layout()
plt.show()

## 2. Tests univaries de drift

### 2.1 Kolmogorov-Smirnov (KS) — pour quanti

Test la difference de **distribution** (CDF). Renvoie une statistique D (distance maximale entre CDFs) et une p-value.
- **D > 0.1** = drift moyen, **D > 0.2** = drift fort (regle de pouce)
- Adapte aux **distributions continues**, **non-parametrique** (pas d'hypothese de normalite)


In [ ]:
def ks_test_drift(ref_col, prod_col):
    stat, pval = stats.ks_2samp(ref_col, prod_col)
    return {"stat": stat, "pvalue": pval, "drifted": pval < 0.05}

print("=== KS test sur features numeriques ===")
print("Reference vs Prod NO drift :")
for col in ["x1", "x2", "constant"]:
    r = ks_test_drift(ref[col], prod_no_drift[col])
    print(f"  {col:10s} : D={r['stat']:.4f}, p={r['pvalue']:.4f}  drift={r['drifted']}")

print("\nReference vs Prod DRIFTED :")
for col in ["x1", "x2", "constant"]:
    r = ks_test_drift(ref[col], prod_drifted[col])
    print(f"  {col:10s} : D={r['stat']:.4f}, p={r['pvalue']:.4f}  drift={r['drifted']}")

**Lecture** : sur prod NO drift, **toutes les features sont OK** (p > 0.05). Sur prod DRIFTED, `x1` est detecte (p < 0.05) — c'est le **vrai drift**.

> ⚠️ **Piege** : avec un gros sample (10k+), le KS detecte des drifts statistiquement significatifs mais **negligeables** en pratique. Toujours regarder la **taille de la statistique D**, pas juste la p-value. Une heuristique : `D > 0.1` pour seuiller.

### 2.2 Chi-square — pour quali

Compare les **frequences** des categories entre ref et prod.


In [ ]:
def chi2_test_drift(ref_col, prod_col):
    """Chi-square independance test : H0 = meme distribution categorielle."""
    # Construire la table de contingence : lignes = categories, colonnes = (ref, prod)
    cats = sorted(set(ref_col.unique()) | set(prod_col.unique()))
    table = pd.DataFrame({
        "ref":  [(ref_col == c).sum() for c in cats],
        "prod": [(prod_col == c).sum() for c in cats],
    }, index=cats)
    stat, pval, dof, expected = stats.chi2_contingency(table.T)
    return {"stat": stat, "pvalue": pval, "table": table, "drifted": pval < 0.05}

print("=== Chi-square sur cat ===")
r1 = chi2_test_drift(ref["cat"], prod_no_drift["cat"])
r2 = chi2_test_drift(ref["cat"], prod_drifted["cat"])
print(f"Ref vs Prod NO drift : chi2={r1['stat']:.2f}, p={r1['pvalue']:.4f}, drift={r1['drifted']}")
print(f"Ref vs Prod DRIFTED  : chi2={r2['stat']:.2f}, p={r2['pvalue']:.4f}, drift={r2['drifted']}")

### 2.3 PSI (Population Stability Index) — standard en banque/credit

PSI est le **standard de fait** en risque credit pour mesurer la stabilite d'une variable. Convention de seuils :
- **PSI < 0.1** : pas de drift
- **0.1 ≤ PSI < 0.2** : drift modere, surveillance
- **PSI ≥ 0.2** : drift majeur, action

Formule : `PSI = Σ_i (prod_i - ref_i) × ln(prod_i / ref_i)` ou les `_i` sont les proportions par bin.


In [ ]:
def psi(ref_col, prod_col, n_bins=10, eps=1e-6):
    """Population Stability Index.

    - Pour quanti : binner via quantiles de ref
    - Pour quali : utiliser les categories directement
    """
    if pd.api.types.is_numeric_dtype(ref_col):
        # Quanti : binner via quantiles de la reference (decile)
        bin_edges = np.unique(np.quantile(ref_col, np.linspace(0, 1, n_bins + 1)))
        if len(bin_edges) < 2:
            return 0.0
        ref_bins = pd.cut(ref_col, bins=bin_edges, include_lowest=True)
        prod_bins = pd.cut(prod_col, bins=bin_edges, include_lowest=True)
        ref_prop = ref_bins.value_counts(normalize=True, sort=False)
        prod_prop = prod_bins.value_counts(normalize=True, sort=False)
    else:
        # Quali
        cats = sorted(set(ref_col.unique()) | set(prod_col.unique()))
        ref_prop = ref_col.value_counts(normalize=True).reindex(cats, fill_value=0)
        prod_prop = prod_col.value_counts(normalize=True).reindex(cats, fill_value=0)

    # Eviter log(0)
    ref_prop = ref_prop.clip(lower=eps)
    prod_prop = prod_prop.clip(lower=eps)
    return float(((prod_prop - ref_prop) * np.log(prod_prop / ref_prop)).sum())


print("=== PSI ===")
for col in ["x1", "x2", "constant", "cat"]:
    p_no = psi(ref[col], prod_no_drift[col])
    p_dr = psi(ref[col], prod_drifted[col])
    interp_no = "OK" if p_no < 0.1 else ("modere" if p_no < 0.2 else "MAJEUR")
    interp_dr = "OK" if p_dr < 0.1 else ("modere" if p_dr < 0.2 else "MAJEUR")
    print(f"  {col:10s} : PSI(no_drift)={p_no:.4f} [{interp_no}]"
          f"   |   PSI(drifted)={p_dr:.4f} [{interp_dr}]")

### 2.4 Jensen-Shannon Divergence

Variante symetrique de la KL divergence, bornee dans [0, log(2)]. Plus stable numeriquement que KL.


In [ ]:
def js_divergence(ref_col, prod_col, n_bins=20):
    """Jensen-Shannon divergence entre les distributions."""
    if pd.api.types.is_numeric_dtype(ref_col):
        bin_edges = np.histogram_bin_edges(np.concatenate([ref_col, prod_col]), bins=n_bins)
        p, _ = np.histogram(ref_col, bins=bin_edges, density=True)
        q, _ = np.histogram(prod_col, bins=bin_edges, density=True)
        p = p / p.sum() + 1e-10
        q = q / q.sum() + 1e-10
    else:
        cats = sorted(set(ref_col.unique()) | set(prod_col.unique()))
        p = np.array([(ref_col == c).mean() for c in cats]) + 1e-10
        q = np.array([(prod_col == c).mean() for c in cats]) + 1e-10
    m = 0.5 * (p + q)
    return 0.5 * (stats.entropy(p, m) + stats.entropy(q, m))

print("=== JS divergence ===")
for col in ["x1", "x2", "constant", "cat"]:
    js_no = js_divergence(ref[col], prod_no_drift[col])
    js_dr = js_divergence(ref[col], prod_drifted[col])
    print(f"  {col:10s} : JS(no_drift)={js_no:.5f}   JS(drifted)={js_dr:.5f}")

## 3. Drift multivarie

Tester chaque feature **independamment** rate les **drifts de correlation** : aucune feature ne drifte individuellement, mais leur **dependence** change.

### 3.1 Domain classifier — methode pragmatique

Idee : entrainer un classifieur a distinguer "vient de ref" vs "vient de prod". Si **AUC ~ 0.5**, pas de drift. Si **AUC >> 0.5**, drift multivarie.


In [ ]:
from sklearn.metrics import roc_auc_score

def domain_classifier_drift(ref_df, prod_df):
    """Train RF a distinguer ref vs prod. Eval via CV (AUC)."""
    # Encoder categorielles
    combined = pd.concat([ref_df.assign(origin=0), prod_df.assign(origin=1)], ignore_index=True)
    combined_enc = pd.get_dummies(combined.drop(columns="origin"), drop_first=True)

    X = combined_enc.values
    y = combined["origin"].values

    # CV pour eviter overfit (AUC sur fold OOS)
    from sklearn.model_selection import cross_val_score
    clf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    auc = cross_val_score(clf, X, y, cv=5, scoring="roc_auc", n_jobs=-1).mean()
    return auc

auc_no = domain_classifier_drift(ref, prod_no_drift)
auc_dr = domain_classifier_drift(ref, prod_drifted)
print(f"Domain classifier AUC :")
print(f"  Ref vs Prod NO drift : {auc_no:.4f}  ({'pas de drift' if auc_no < 0.55 else 'drift detecte'})")
print(f"  Ref vs Prod DRIFTED  : {auc_dr:.4f}  ({'pas de drift' if auc_dr < 0.55 else 'drift detecte'})")

**Bonus du domain classifier** : on peut inspecter les **feature importances** pour savoir QUELLES features driftent le plus.


In [ ]:
# Importance globale du drift par feature
combined = pd.concat([ref.assign(origin=0), prod_drifted.assign(origin=1)], ignore_index=True)
combined_enc = pd.get_dummies(combined.drop(columns="origin"), drop_first=True)
X = combined_enc.values
y = combined["origin"].values

clf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X, y)
imps = pd.Series(clf.feature_importances_, index=combined_enc.columns).sort_values(ascending=False)
print("Top features qui driftent :")
print(imps.head(10))

## 4. Drift sur la cible (label drift)

Si on a `y` en prod (avec un delay), on peut comparer `P(Y_ref)` vs `P(Y_prod)`. Souvent un signal precoce de probleme business.


In [ ]:
# Simuler des labels avec drift de prior
np.random.seed(SEED)
y_ref = np.random.binomial(1, p=0.3, size=len(ref))
y_prod_no = np.random.binomial(1, p=0.3, size=len(prod_no_drift))
y_prod_dr = np.random.binomial(1, p=0.5, size=len(prod_drifted))   # prior shift

print(f"P(Y=1) ref         : {y_ref.mean():.3f}")
print(f"P(Y=1) prod no     : {y_prod_no.mean():.3f}")
print(f"P(Y=1) prod drift  : {y_prod_dr.mean():.3f}  <- LABEL DRIFT")

# Test chi2 de label drift
table = pd.DataFrame({
    "ref":  [np.sum(y_ref == 0), np.sum(y_ref == 1)],
    "prod_dr": [np.sum(y_prod_dr == 0), np.sum(y_prod_dr == 1)],
}, index=["y=0", "y=1"])
stat, pval, _, _ = stats.chi2_contingency(table.T)
print(f"\nChi-square label drift : chi2={stat:.2f}, p={pval:.4f}, drift={'OUI' if pval < 0.05 else 'NON'}")

## 5. Concept drift — degradation de performance

Concept drift = `P(Y|X)` change. Detectable **uniquement** si on a la verite terrain en prod (Y_prod). On compare la perf sur des fenetres glissantes.

> ⚠️ **Cas frequent** : on n'a `Y_prod` qu'avec un delay (1 mois pour churn, 6 mois pour credit). Pour pallier, **NannyML** (cf. `MLOps_Drift_Monitoring.ipynb` section NannyML — pas couvert ici) **estime la perf sans Y vrai**.


In [ ]:
# Demo : entrainer modele sur ref, mesurer perf sur differentes "semaines" de prod
np.random.seed(SEED)

# Modele entraine sur ref + y_ref
clf_drift = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
X_ref_enc = pd.get_dummies(ref, drop_first=True)
clf_drift.fit(X_ref_enc, y_ref)

# Predire sur prod_no_drift et prod_drifted, on simule la verite terrain qui drifte aussi
X_prod_no_enc = pd.get_dummies(prod_no_drift, drop_first=True).reindex(columns=X_ref_enc.columns, fill_value=0)
X_prod_dr_enc = pd.get_dummies(prod_drifted, drop_first=True).reindex(columns=X_ref_enc.columns, fill_value=0)

# Fenetres glissantes
window = 500
def perf_window(X, y, w):
    aucs = []
    for i in range(0, len(X) - w, w):
        proba = clf_drift.predict_proba(X.iloc[i:i+w])[:, 1]
        aucs.append(roc_auc_score(y[i:i+w], proba))
    return aucs

aucs_no = perf_window(X_prod_no_enc, y_prod_no, window)
aucs_dr = perf_window(X_prod_dr_enc, y_prod_dr, window)

print(f"Perf (AUC) en prod NO drift, fenetres de {window} :")
print(f"  AUCs   : {[f'{a:.3f}' for a in aucs_no]}")
print(f"  Mean   : {np.mean(aucs_no):.3f}")
print(f"\nPerf (AUC) en prod DRIFTED, fenetres de {window} :")
print(f"  AUCs   : {[f'{a:.3f}' for a in aucs_dr]}")
print(f"  Mean   : {np.mean(aucs_dr):.3f}")

## 6. Evidently AI — reports HTML automatises

[Evidently](https://docs.evidentlyai.com/) est la lib **standard** pour produire des reports de drift en quelques lignes. Sortie HTML interactive + JSON pour pipelines.

Installation : `uv add --group mlops evidently` (ou `pip install -q evidently`).


In [ ]:
# pip install -q evidently
# import evidently
# print(f"Evidently version : {evidently.__version__}")
# from evidently import Report
# from evidently.metric_preset import DataDriftPreset, DataQualityPreset, ClassificationPreset

# # Combine en un dataset evidently
# report = Report(metrics=[
#     DataDriftPreset(),
#     DataQualityPreset(),
# ])
# report.run(reference_data=ref, current_data=prod_drifted)
# report.save_html("drift_report.html")
# # Vue notebook :
# # report.show()

# # En JSON pour pipeline alerting
# # import json
# # data = report.as_dict()
# # if data["metrics"][0]["result"]["dataset_drift"]:
# #     send_alert("Drift detected!")
print("Evidently : decommenter pour generer un rapport HTML interactif.")
print("Code pret a exec une fois evidently installe.")

### 6.1 Evidently TestSuite (assertions binaires)

Au lieu d'un report descriptif, on declare des **tests** binaires (a passer ou pas). Bon pour CI/CD :


In [ ]:
# from evidently import TestSuite
# from evidently.tests import (
#     TestNumberOfMissingValues, TestShareOfMissingValues,
#     TestColumnDrift, TestColumnQuantile,
# )

# suite = TestSuite(tests=[
#     TestShareOfMissingValues(lt=0.05),
#     TestColumnDrift(column_name="x1", stattest="ks", stattest_threshold=0.05),
#     TestColumnDrift(column_name="cat", stattest="chisquare", stattest_threshold=0.05),
#     TestColumnQuantile(column_name="x2", quantile=0.99, lt=20),
# ])
# suite.run(reference_data=ref, current_data=prod_drifted)
# suite.save_html("tests.html")
# # suite.as_dict() pour parser dans CI
print("TestSuite Evidently : code pret pour CI/CD")

## 7. River — online learning et drift natif

[River](https://riverml.xyz/) est la lib Python de reference pour **streaming ML** (one sample at a time). Inclut des detecteurs de drift natifs.

### 7.1 ADWIN — Adaptive Windowing

ADWIN (Bifet & Gavalda 2007) detecte en ligne un changement dans une serie de scalaires. Adapte sa fenetre dynamiquement.


In [ ]:
# pip install -q river
# from river import drift

# detector = drift.ADWIN(delta=0.002)

# # Simulation : 1000 valeurs stables a 0.5 puis 1000 valeurs a 0.8 (drift)
# np.random.seed(SEED)
# stream = np.concatenate([
#     np.random.binomial(1, 0.5, 1000),
#     np.random.binomial(1, 0.8, 1000),
# ])

# drift_detected_at = []
# for i, x in enumerate(stream):
#     detector.update(x)
#     if detector.drift_detected:
#         drift_detected_at.append(i)

# print(f"Drifts detectes aux indices : {drift_detected_at}")
# # Sortie typique : autour de 1000-1100 (ou la distribution change reellement)
print("ADWIN : decommenter pour run (necessite river installe)")

### 7.2 Page-Hinkley — alternative

Detecte un changement de **moyenne** dans un signal. Plus simple, parfois plus rapide qu'ADWIN.


In [ ]:
# from river import drift

# ph = drift.PageHinkley(min_instances=30, delta=0.005, threshold=50)
# # ... usage identique

# # Comparatif :
# # - ADWIN : adaptive window, detection plus robuste mais plus lent
# # - PageHinkley : tres rapide, pour changements de moyenne brusques
# # - KSWIN : Kolmogorov-Smirnov sur fenetre, multivarie possible
# # - DDM (Drift Detection Method) : pour erreurs de classifier
print("Page-Hinkley : alternative legere a ADWIN")

## 8. Patterns d'architecture monitoring en production

### Architecture batch typique

```
[Inference logs] (S3/BigQuery/Snowflake)
        │
        ▼
[Job batch quotidien (Airflow/Prefect)]
        │
        ├──> Evidently report HTML → S3 + dashboard
        ├──> Tests assertions → if fail → PagerDuty/Slack
        └──> Performance estimation (NannyML)
                │
                ▼
        [Trigger retraining si dégradation]
```

### Strategies de retraining

| Strategie | Quand l'utiliser |
|---|---|
| **Periodique** (chaque mois) | Drift previsible (saisonnalite) |
| **Triggered on drift** | Detection automatique → retrain |
| **Performance-based** | Metrique business franchit un seuil |
| **Continuous** (online) | Stream + river / Vowpal Wabbit |
| **Champion/Challenger** | v2 trained en shadow, switch si meilleur |

### Anti-spam : eviter le bruit

| Probleme | Solution |
|---|---|
| P-value < 0.05 sur 100 features → alerte sur 5 par hasard | Bonferroni correction : `α / N` ou FDR (Benjamini-Hochberg) |
| Alerte sur 1 spike isole | Window-based : drift maintenu sur N batches consecutifs |
| Drift stat sans impact business | Filtrer par taille d'effet (PSI > 0.2, D > 0.1) |
| Alerte sans contexte | Inclure direction du drift, valeur ref vs prod, recommandation |

### Bonferroni correction


In [ ]:
def bonferroni_correction(pvalues, alpha=0.05):
    """Renvoie les indices significatifs apres correction de Bonferroni."""
    n = len(pvalues)
    threshold = alpha / n
    return [i for i, p in enumerate(pvalues) if p < threshold], threshold

# Demo : on teste 30 features, certaines avec drift, d'autres juste bruit
np.random.seed(SEED)
pvalues = list(np.random.uniform(0, 1, 27)) + [0.001, 0.003, 0.04]  # 3 vrais drifts
sig_naif = [i for i, p in enumerate(pvalues) if p < 0.05]
sig_bonf, thr = bonferroni_correction(pvalues, alpha=0.05)

print(f"Sans correction (alpha=0.05) : {len(sig_naif)} features significatives")
print(f"Avec Bonferroni (alpha/N={thr:.4f}) : {len(sig_bonf)} features significatives")
print(f"\nLes vrais drifts (indices 27, 28, 29) sont-ils retenus apres Bonferroni ?")
print(f"  -> retenus : {sig_bonf}")

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Alerter sur chaque p-value < 0.05 | Filtrer par taille d'effet (PSI > 0.2, D > 0.1, KS > 0.1) |
| Monitorer juste l'accuracy globale | Par segment (region, plage horaire, version client) |
| Detecter le drift apres 6 mois | Window glissante quotidienne |
| Comparer ref ancienne vs prod sans tenir compte de la saisonnalite | Reference glissante / year-over-year |
| Pas tester le shape `X.shape` | Schema drift = bug silencieux (colonne renommee en amont) |
| Retraining auto sans validation | Pipeline : retrain → validate → canary → 100% |
| KS test sur 1M lignes : tout est "significatif" | Toujours D + p ensemble |
| Alerter en multi-channel (mail+Slack+PagerDuty) sans deduplication | Tag les drifts unifies par hash |
| Confondre p-value 0.04 et 0.000001 | Magnitude compte : passer p-values en log scale |
| Ignorer le drift cible (`Y`) | Au moins aussi important que feature drift |

### Diagnostic checklist
- [ ] **Schema** : meme colonnes, memes dtypes ?
- [ ] **Coverage** : pas de NaN explosion ?
- [ ] **Univariate drift** : KS / Chi-2 par feature
- [ ] **Multivariate drift** : domain classifier
- [ ] **Label drift** : P(Y) stable ?
- [ ] **Concept drift** : si Y_prod disponible, perf stable ?
- [ ] **Business KPI** : metriques produit (CTR, churn, revenue) en accord ?


## 10. References

### Documentation officielle
- **Evidently AI** : https://docs.evidentlyai.com/
- **NannyML** (perf estimation sans Y) : https://nannyml.readthedocs.io/
- **Alibi Detect** : https://docs.seldon.io/projects/alibi-detect/
- **River** : https://riverml.xyz/
- **WhyLabs** / **Arize** / **Fiddler** : SaaS observabilite ML

### Papers fondateurs
- Gama et al. (2014). *A Survey on Concept Drift Adaptation*.
- Bifet & Gavalda (2007). *Learning from Time-Changing Data with Adaptive Windowing* (ADWIN).
- Page (1954). *Continuous Inspection Schemes* (Page-Hinkley).
- Wasserstein et al. (2019). *Moving to a World Beyond "p < 0.05"* (ASA position).

### Voir aussi (collection)
- [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb) — tracking expe
- [MLOps_Model_Serving.ipynb](./MLOps_Model_Serving.ipynb) — serving prod
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — metriques d'evaluation
- [MLOps_Pipelines_Airflow_Prefect.ipynb](./MLOps_Pipelines_Airflow_Prefect.ipynb) — orchestration


<!-- ENRICH_2025_V1 -->

## 🔥 Outils de drift monitoring 2024-2025

### 1. **NannyML** — Performance estimation SANS labels

**Probleme reel** : en prod, `y_true` arrive avec un delay (semaines pour churn, mois pour credit). NannyML estime la **performance** sans `y_true` via :

- **CBPE** (Confidence-Based Performance Estimation) pour classif
- **DLE** (Direct Loss Estimation) pour regression

```python
# pip install nannyml
import nannyml as nml

estimator = nml.CBPE(
    y_pred_proba="proba",
    y_pred="prediction",
    y_true="ground_truth",
    metrics=["roc_auc", "f1"],
    chunk_size=1000,
)
estimator.fit(reference_df)
estimated_perf = estimator.estimate(analysis_df)
estimated_perf.plot().show()    # courbe perf estimee avec CI
```

### 2. **Alibi Detect** (Seldon)

Drift detection multivarie avec **MMD** (Maximum Mean Discrepancy), **CVM**, **FET**, drift sur embeddings/images.

```python
# pip install alibi-detect
from alibi_detect.cd import MMDDrift
mmd = MMDDrift(X_ref, p_val=0.05, n_permutations=100)
pred = mmd.predict(X_test)
print(pred["data"]["is_drift"])
```

### 3. **WhyLabs / Arize / Fiddler / Aporia** (SaaS)

Plateformes managed observability ML. **Production-grade** : alerting, dashboards, root cause analysis automatique.

### 4. **River** — online learning + drift natif

Pour le **streaming** :
```python
from river import drift

detector = drift.ADWIN(delta=0.002)
for x in stream:
    detector.update(x)
    if detector.drift_detected:
        retrain_model()
```

## 🔥 Patterns avances 2025

### **Concept drift detection sans labels**

NannyML : CBPE → estime la **degradation de la confidence** du modele. Si confidence baisse → potentiel concept drift.

### **Multivariate drift = domain classifier**

Entrainer un classifieur **ref vs prod**. Si AUC >> 0.5 → drift detectable multivarie (meme si chaque feature individuelle n'a pas drifte).

### **Drift segment-aware**

Surveiller par segment (region, plage horaire, version client) — souvent un segment specifique drifte (nouveau marche), invisible en agrege.

### **Outlier-aware monitoring**

Combiner detection drift + detection outliers (Isolation Forest) → catch les **input out-of-distribution** AVANT qu'ils contaminent le modele.

## 🔥 Anti-spam — corrections statistiques

- **Bonferroni** : `α / N` pour N tests simultanes (sinon 5% de faux positifs)
- **Benjamini-Hochberg** (FDR) : moins conservatif, controle le False Discovery Rate
- **Window-based** : drift maintenu sur N batches consecutifs (pas 1 spike)
- **PSI > 0.2** comme seuil business (au lieu de p < 0.05 mecanique)

## 💡 Stack monitoring recommande 2025

| Echelle | Stack |
|---|---|
| **Solo / dev** | Evidently (HTML reports) + sklearn drift simple |
| **Equipe** | Evidently + NannyML + Airflow daily |
| **Production large** | Arize / WhyLabs / Aporia (SaaS) |
| **Streaming** | River + Kafka + Prometheus alerting |
| **On-prem strict** | Evidently self-hosted + Grafana |

## 💡 Recap "quand quoi"

| Symptome | Outil |
|---|---|
| "Mes features ressemblent moins au train" | Evidently `DataDriftPreset` |
| "Mes labels sont en retard" | NannyML CBPE/DLE |
| "Drift multivarie subtil" | Domain classifier + Alibi MMD |
| "Stream temps reel" | River + ADWIN |
| "Production critique avec SLA" | SaaS (Arize / WhyLabs) |
